# 남은 pose8 — 고정 확장 cache·새 작업
기존269 중 Colab257개와 local4개를 확보했습니다. 남은8개만 고정 posePrefixTargets8_v1으로 한 번 실행합니다.
새 CPU runtime에서 작은 source/input ZIP 두 개를 준비하고 인증·Drive mount 후 확인된 cache 경로 한 개만 읽습니다. CACHE_DRIVE_PATH는 root가 확인하기 전 빈값을 유지합니다.
확장 cache250718584bytes/SHA e8516bf6214bac6253551ec778dad6a57a3c6d6657af4200cab12994713f996f를 readonly로1MiB씩 복사하고 local 전체 SHA/input_spec을 검증합니다. 준비300초 협력·disk256MiB·metadata2MiB는 원본 prepare와 별개입니다. 실제 FUSE wire량이나 단일 blocked read의 즉시 timeout은 보장하지 않습니다.
새 job pose_missing8_001은 원본body192MiB/1916HTTP/1200초 이하, 논리read352MiB/decode2400MiB, 일반출력40MiB/결과ZIP41MiB입니다. 기존269_v1/67/일반 예산은 바꾸지 않습니다.
소비된 workspace는 재실행하지 않습니다. 실패 partial·보고서·기존회수ZIP을 보존하며 자동삭제/반복/cold fallback은 없습니다. 결과ZIP과 확장cache ZIP은 각각 영구보관합니다. 원격 실행과 학습은 아직 하지 않았습니다.


In [ ]:
# root가 커밋 후 source/입력 SHA와 revision·job 경로를 고정합니다.
SOURCE_ZIP_SHA256 = ""
SOURCE_REVISION = ""
NOTEBOOK_REVISION = ""
INPUT_ZIP_SHA256 = "e665b671846faee1e486c0369c1a9ff0d71eee0e3a5be7068c529a23c0f27762"
TARGETS_SHA256 = "7def9e3b9e15575df4bf7253b519f49814af94bfcd2fa20bee2096ada2c44c6c"
CACHE_ZIP_SHA256 = "e8516bf6214bac6253551ec778dad6a57a3c6d6657af4200cab12994713f996f"
JOB_ID = "pose_missing8_001"
SOURCE_ROOT = "/content/pose_missing8_001-source"
INPUT_ROOT = "/content/pose_missing8_001-inputs"
CACHE_INPUT = "/content/pose_missing8_001-cache.zip"
WORKSPACE = "/content/pose_missing8_001-work"
RECOVERY_ZIP = "/content/pose_missing8_drive_001_preparation_result.zip"
# root/탐색 세션이 실제 파일을 확인한 뒤 ready에 정확한 경로를 고정합니다.
CACHE_DRIVE_PATH = ""  # 자동 검색·추측·다른 cache fallback 금지
CACHE_STAGE_REPORT = "/content/pose_missing8_cache_stage.json"


In [ ]:
import hashlib, io, json, re, stat, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath
from google.colab import files
assert re.fullmatch("[a-f0-9]{64}", SOURCE_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{64}", INPUT_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{64}", TARGETS_SHA256)
assert re.fullmatch("[a-f0-9]{64}", CACHE_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{40}", SOURCE_REVISION)
assert not Path(WORKSPACE).exists() and not Path(WORKSPACE).is_symlink(), "소비된 job: 준비/scan 재실행 금지"
import psutil, shutil, time
assert psutil.virtual_memory().available >= 2*1024**3, "가용 RAM2GiB 필요"
assert shutil.disk_usage(Path(SOURCE_ROOT).parent).free >= 2*1024**3, "임시 disk2GiB 필요"
print("1/2: 고정 source ZIP을 선택하세요.")
source_uploaded = files.upload()
print("2/2: 고정 input ZIP을 선택하세요.")
input_uploaded = files.upload()
local_started = time.monotonic()
def selected(uploaded, pin, label):
    rows = [b for b in uploaded.values() if hashlib.sha256(b).hexdigest() == pin]
    assert len(rows) == 1, label + " ZIP 누락 또는 고정 SHA 불일치"
    return rows[0]
def inspect_zip(raw, maximum):
    assert len(raw) <= maximum, "압축 ZIP 크기 상한"
    payloads, directories, names = {}, set(), set()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        rows = z.infolist()
        assert len(rows) <= 2048 and sum(i.file_size for i in rows) <= maximum, "ZIP 해제 상한"
        for item in rows:
            name = item.filename.rstrip("/")
            path = PurePosixPath(name)
            assert name and name == path.as_posix() and not path.is_absolute()
            assert ".." not in path.parts and "\\" not in name and all(":" not in part for part in path.parts), "ZIP 경로 오류"
            assert name.casefold() not in names, "ZIP 중복 경로"
            names.add(name.casefold())
            kind = stat.S_IFMT(item.external_attr >> 16)
            assert kind in (0, stat.S_IFDIR if item.is_dir() else stat.S_IFREG), "ZIP 링크/특수파일 금지"
            for parent in path.parents:
                if str(parent) != ".": directories.add(parent.as_posix())
            data = z.read(item)  # 모든 member CRC를 추출 전에 검사
            assert len(data) == item.file_size
            if item.is_dir(): directories.add(name)
            else: payloads[name] = data
    assert not set(payloads) & directories, "ZIP 파일/폴더 충돌"
    return payloads, directories
# source/input 두 ZIP을 검증한 뒤 준비 경로를 만듭니다.
source_data = inspect_zip(selected(source_uploaded, SOURCE_ZIP_SHA256, "source"), 64*1024**2)
input_data = inspect_zip(selected(input_uploaded, INPUT_ZIP_SHA256, "input"), 8*1024**2)
assert source_data[0]["SOURCE_REVISION.txt"].decode("utf-8").strip() == SOURCE_REVISION
input_config = json.loads(input_data[0]["config.json"])
targets = json.loads(input_data[0]["targets.json"])
assert len(targets) == 8 and targets == input_config["job"]["exact_targets"] == input_config["job"]["includes"]
assert hashlib.sha256(json.dumps(targets, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest() == TARGETS_SHA256 == input_config["targets_content_sha256"]
assert CACHE_ZIP_SHA256 == input_config["cache_zip_sha256"]
assert input_config["cache_zip_bytes"] == 250718584
def existing_matches(destination, contents):
    target = Path(destination).absolute()
    for part in (target, *target.parents):
        assert not part.is_symlink() and not getattr(part, "is_junction", lambda:False)(), "추출 경로 링크 금지"
    if not target.exists(): return False
    assert target.is_dir(), "추출 대상이 디렉터리가 아님"
    files_seen, dirs_seen = set(), set()
    for count, path in enumerate(target.rglob("*")):
        assert count < 4096 and not path.is_symlink() and not getattr(path, "is_junction", lambda:False)()
        relative = path.relative_to(target).as_posix()
        if path.is_dir(): dirs_seen.add(relative)
        else:
            assert path.is_file() and relative in contents[0], "기존 추출본의 다른 파일"
            expected = contents[0][relative]
            assert path.stat().st_size == len(expected) and path.read_bytes() == expected, "기존 추출본 바이트 불일치"
            files_seen.add(relative)
    assert files_seen == set(contents[0]) and dirs_seen == contents[1], "기존 추출본 파일/폴더 집합 불일치"
    return True
source_path, input_path = Path(SOURCE_ROOT).absolute(), Path(INPUT_ROOT).absolute()
assert not source_path.is_relative_to(input_path) and not input_path.is_relative_to(source_path)
source_exists = existing_matches(SOURCE_ROOT, source_data)
input_exists = existing_matches(INPUT_ROOT, input_data)
def materialize(destination, contents, exists):
    if exists: return  # 정확히 같은 기존 추출본만 읽기 재사용
    target = Path(destination)
    target.mkdir(parents=True, exist_ok=False)
    for name in sorted(contents[1], key=lambda s:len(PurePosixPath(s).parts)):
        (target/name).mkdir(exist_ok=True)
    for name, data in contents[0].items():
        with (target/name).open("xb") as stream: stream.write(data)
materialize(SOURCE_ROOT, source_data, source_exists)
materialize(INPUT_ROOT, input_data, input_exists)
assert time.monotonic()-local_started <= 60, "작은 ZIP 로컬 준비60초 제한"
sys.dont_write_bytecode = True
sys.path.insert(0, SOURCE_ROOT)
del source_uploaded, input_uploaded, source_data, input_data


In [ ]:
# 큰 cache는 브라우저 upload하지 않습니다. 인증/mount 뒤 명시한 Drive 파일만 읽습니다.
import os, sys
from pathlib import Path
assert not Path(WORKSPACE).exists(), "소비된 job: 준비/실행 재호출 금지"
assert CACHE_DRIVE_PATH and Path(CACHE_DRIVE_PATH).is_absolute(), "root가 확인한 정확한 Drive 경로 필요"
for name,module in tuple(sys.modules.items()):
    if name == "rewind_archive" or name.startswith("rewind_archive."):
        loaded=getattr(module,"__file__",None)
        assert loaded and Path(loaded).resolve().is_relative_to(Path(SOURCE_ROOT).resolve()), "이전 코드가 로드됨: 새 CPU 런타임에서 시작하세요"
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "실제 Drive mount 확인 실패"
from rewind_archive.cache_staging import stage_pose8_cache
STAGED=stage_pose8_cache(CACHE_DRIVE_PATH,CACHE_INPUT,source_root=SOURCE_ROOT,input_root=INPUT_ROOT,
    workspace=WORKSPACE,report_path=CACHE_STAGE_REPORT,readonly_root="/content/drive",local_root="/content")
print(STAGED["report"])


In [ ]:
# Cache 준비가 검증 완료된 보고서일 때만 기존 preflight로 넘어갑니다.
import json
from pathlib import Path
assert not Path(WORKSPACE).exists(), "소비된 job: 다시 실행하지 마세요"
stage_info=json.loads(Path(CACHE_STAGE_REPORT).read_bytes())
assert stage_info["identity"]["expected_sha256"] == CACHE_ZIP_SHA256
assert stage_info["identity"]["target"] == str(Path(CACHE_INPUT).resolve())
assert stage_info["attempts"][-1]["status"] in ("verified_existing_staging","verified_copied_staging")
# 네트워크 전, 고정 source·cache policy·목표·예산을 검사합니다.
import json, sys
from pathlib import Path
sys.dont_write_bytecode = True
for name, module in tuple(sys.modules.items()):
    if name == "rewind_archive" or name.startswith("rewind_archive."):
        loaded = getattr(module, "__file__", None)
        assert loaded and Path(loaded).resolve().is_relative_to(Path(SOURCE_ROOT).resolve()), "이전 rewind_archive가 로드됨: 새 CPU 런타임에서 다시 준비하세요"
sys.path.insert(0, SOURCE_ROOT)
from rewind_archive.workflow import make_preflight
from rewind_archive.prefix_cache import input_spec
from rewind_archive.cache_policy import TAIL352_POLICY
config = json.loads((Path(INPUT_ROOT)/"config.json").read_bytes())
JOB = config["job"]
JOB["job_id"] = JOB_ID
JOB["cache_input"] = input_spec(CACHE_INPUT, policy=TAIL352_POLICY)
PREFLIGHT = make_preflight(JOB, {})
print(PREFLIGHT["summary"])


In [ ]:
# 1회만 실행합니다. 중단/실패 후에는 회수 셀만 실행합니다.
import json, sys
from pathlib import Path
assert not Path(WORKSPACE).exists(), "소비된 job: 재실행 금지"
# 인증/mount는 앞선 cache 준비 셀에서 완료했습니다.
import google.auth
import requests
from google.auth.transport.requests import AuthorizedSession, Request
from rewind_archive.request_policy import validate_no_hidden_retries
from urllib3.util.retry import Retry
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
def session_factory():
    auth_http = requests.Session()
    try:
        auth_http.trust_env = False
        for adapter in auth_http.adapters.values():
            adapter.max_retries = Retry(total=0, connect=0, read=0, redirect=0, status=0)
        session = AuthorizedSession(credentials, max_refresh_attempts=0, auth_request=Request(session=auth_http))
    except BaseException:
        auth_http.close()
        raise
    close_outer = session.close
    def close_all():
        try:
            close_outer()
        finally:
            auth_http.close()
    session.close = close_all
    try:
        session.trust_env = False
        for adapter in session.adapters.values():
            adapter.max_retries = Retry(total=0, connect=0, read=0, redirect=0, status=0)
        # 초기 metadata 호출 전 내부 auth transport까지 검사한다.
        validate_no_hidden_retries(session)
    except BaseException:
        session.close()
        raise
    return session
from rewind_archive.workflow import run_workflow_job
OUTPUTS = Path(WORKSPACE)/"outputs"
OUTPUTS.mkdir(parents=True, exist_ok=False)
RESULT_ROOT, RECORD = run_workflow_job(JOB, PREFLIGHT, {}, OUTPUTS, session_factory)
print({k:RECORD.get(k) for k in ("status","stop_reason","completed_targets","missing_targets","targets_complete")})


In [ ]:
# 독립 회수: 인증/원본 요청/run은 다시 호출하지 않습니다.
import hashlib, json, sys, zipfile
from pathlib import Path
from google.colab import files
def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while data := stream.read(65536): h.update(data)
    return h.hexdigest()
job_root = Path(WORKSPACE)/"outputs"/("archive_job_"+JOB_ID)
stage_report = Path(CACHE_STAGE_REPORT)
stage_raw = None
if stage_report.is_file():
    assert not stage_report.is_symlink() and stage_report.stat().st_size <= 65536
    stage_raw = stage_report.read_bytes()
    stage_info = json.loads(stage_raw)
    assert stage_info["identity"]["expected_sha256"] == CACHE_ZIP_SHA256
    assert stage_info["identity"]["expected_bytes"] == 250718584
if not (job_root/"job.json").is_file():
    target = Path(RECOVERY_ZIP)
    if stage_raw is not None:
        stage_pin = hashlib.sha256(stage_raw).hexdigest()
        target = target.with_name(target.stem.removesuffix("_preparation_result")+"_stage_"+stage_pin[:16]+"_preparation_result.zip")
    assert target.name.endswith("_preparation_result.zip")
    if not target.exists():
        info = {"job_id":JOB_ID, "targets_sha256":TARGETS_SHA256, "source_revision":SOURCE_REVISION,
            "notebook_revision":NOTEBOOK_REVISION, "status":"preparation_or_auth_failed_execution_unverified",
            "job_report_present":False, "train_ready":False}
        with zipfile.ZipFile(target, "x", zipfile.ZIP_STORED) as z:
            z.writestr("recovery.json", json.dumps(info, ensure_ascii=False, sort_keys=True))
            if stage_raw is not None: z.writestr("cache_stage_report.json", stage_raw)
    with zipfile.ZipFile(target) as z:
        assert z.testzip() is None
        info = json.loads(z.read("recovery.json"))
        if stage_raw is not None: assert z.read("cache_stage_report.json") == stage_raw, "현재 준비 보고서와 회수 ZIP 불일치"
        assert info["job_id"] == JOB_ID and info["targets_sha256"] == TARGETS_SHA256
    print("준비 결과 ZIP", str(target), file_sha(target))
    files.download(str(target))
else:
    assert (Path(SOURCE_ROOT)/"SOURCE_REVISION.txt").read_text().strip() == SOURCE_REVISION
    sys.dont_write_bytecode = True
    sys.path.insert(0, SOURCE_ROOT)
    from rewind_archive.workflow import review_run, save_result_zip, save_cache_zip
    from rewind_archive.contracts import PreparationError
    if stage_raw is not None:
        stage_copy=job_root/"pilot"/"cache_stage_report.json"
        stage_copy.parent.mkdir(exist_ok=True)
        if not stage_copy.exists():
            with stage_copy.open("xb") as stream: stream.write(stage_raw)
    checked = review_run(job_root)
    extra = job_root/"pilot"/"pose8_review.json"
    extra.parent.mkdir(exist_ok=True)
    if not extra.exists():
        with extra.open("x", encoding="utf-8") as stream: json.dump(checked, stream, ensure_ascii=False, indent=2)
    cache = None
    if (job_root/"pilot"/"cache"/"manifest.json").is_file():
        try:
            cache, cache_report = save_cache_zip(job_root)
            print({"cache_zip":str(cache), "cache_export":cache_report})
        except PreparationError as exc:
            print({"cache_export_failed":exc.reason, "original_cache_preserved":True})
    # 결과 ZIP은 cache body와 별도이며 JSON·hash·missing·종료 이유·관측목록을 포함합니다.
    result = save_result_zip(job_root)
    print("결과 ZIP", str(result), file_sha(result))
    files.download(str(result))
    if cache is not None:
        print("확장 cache ZIP", str(cache), file_sha(cache))
        files.download(str(cache))
